# 04 - Cropland-conversion labels for grass pixels

**Question:** does a patch of grass (pasture or natural grassland) look like land that ends up being farmed as cropland?

**Label (observed outcome, not a vigor proxy).** For a base year `t0`:

- Candidate pixels: stable grass (MapBiomas class 12 Grassland or 15 Pasture) in every year of the feature window `t0-W+1 .. t0`.
- **Positive (1, `converted`)**: cropland in years `t0+K-1` and `t0+K` (two consecutive years, to drop one-year noise).
- **Negative (0, `stayed_grass`)**: grass in every year `t0+1 .. t0+K`.
- Everything else (forest, urban, mosaic of uses, one-off crop flicker...) is masked out as ambiguous.

**Features use data up to `t0` only** (AlphaEarth embeddings for `t0-W+1 .. t0`, static terrain, rainfall, and the cropland share of the surrounding 2 km at `t0`), so nothing from the label window leaks in.

**Caveat.** A negative means "was not converted in K years", not "unsuitable" (ownership, distance, legal protection and economics also matter). Treat negatives as unlabeled-leaning.

Output: `data/processed/conversion_dataset.npz` and `data/processed/conversion_points.csv`.

In [1]:
import time
import ee
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

EE_PROJECT_ID = 'projetograssquality'  # same project as 01_dados.ipynb / app.py
ee.Authenticate()


Successfully saved authorization token.


In [2]:
ee.Initialize(project=EE_PROJECT_ID)
print('Earth Engine initialized.')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized.


In [3]:
# ---- Configuration -------------------------------------------------------------
AOI = ee.Geometry.Rectangle([-49.6, -16.9, -48.9, -16.3])   # same AOI as 01_dados.ipynb (Goiania)

LULC_ASSET = 'projects/mapbiomas-public/assets/brazil/lulc/v1'   # ImageCollection, band 'classification', prop 'year', 1985-2024
AE_ASSET   = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'              # bands A00..A63

GRASS_CLASSES = [12, 15]                   # 12 Grassland, 15 Pasture
CROP_CLASSES  = [18, 19, 20, 39, 40, 41, 62]   # Agriculture, temporary crop, sugarcane, soybean, rice, other temporary, cotton
# Perennial crops (36 perennial, 46 coffee, 47 citrus, 48 other perennial) are left out on purpose: "cropland" = temporary crops here.
# Append them to CROP_CLASSES if you decide otherwise.

BASE_YEARS = [2019, 2020]   # t0
WINDOW     = 3              # feature window in years: t0-W+1 .. t0  (AlphaEarth starts in 2017, so t0 >= 2019 for W=3)
HORIZON    = 4              # K: label year = t0 + K  (must be <= 2024 -> t0 <= 2020 for K=4)

POINTS_PER_STRATUM_PER_YEAR = 250   # 4 strata: {natural grassland, pasture} x {stayed, converted}
CROP_NEIGHBORHOOD_M = 2000
GRID_DEG = 0.05                     # ~5.5 km spatial blocks for the train/val/test split (same as 01_dados)

assert all(t0 - WINDOW + 1 >= 2017 for t0 in BASE_YEARS), 'AlphaEarth annual starts in 2017'
assert all(t0 + HORIZON <= 2024 for t0 in BASE_YEARS), 'MapBiomas LULC v1 ends in 2024 (check for newer years)'

In [4]:
lulc_col = ee.ImageCollection(LULC_ASSET)

def lulc(year):
    return lulc_col.filter(ee.Filter.eq('year', year)).first().select('classification')

def isin(img, codes):
    return img.remap(codes, [1] * len(codes), 0)

def all_years(fn, years):
    out = fn(years[0])
    for y in years[1:]:
        out = out.And(fn(y))
    return out

def build_label_image(t0):
    hist_years   = list(range(t0 - WINDOW + 1, t0 + 1))
    future_years = list(range(t0 + 1, t0 + HORIZON + 1))

    is_grass = lambda y: isin(lulc(y), GRASS_CLASSES)
    is_crop  = lambda y: isin(lulc(y), CROP_CLASSES)

    stable_grass = all_years(is_grass, hist_years)
    stayed       = all_years(is_grass, future_years)
    converted    = is_crop(t0 + HORIZON).And(is_crop(t0 + HORIZON - 1))

    label = converted.rename('label')                       # 1 = converted, 0 = (stayed or ambiguous)
    valid = stable_grass.And(stayed.Or(converted))          # drop ambiguous transitions
    grass_type = lulc(t0).eq(15).rename('grass_type')       # 0 = natural grassland (12), 1 = pasture (15)
    strat = label.add(grass_type.multiply(2)).rename('strat')  # 0..3
    return label.addBands(grass_type).addBands(strat).updateMask(valid)

In [5]:
rows, next_id = [], 0
for t0 in BASE_YEARS:
    img = build_label_image(t0)
    fc = img.stratifiedSample(
        numPoints=0,
        classBand='strat',
        region=AOI,
        scale=30,
        seed=SEED + t0,
        classValues=[0, 1, 2, 3],
        classPoints=[POINTS_PER_STRATUM_PER_YEAR] * 4,
        geometries=True,
        tileScale=4,
    )
    feats = fc.getInfo()['features']
    print(f'base year {t0}: {len(feats)} points')
    for f in feats:
        lon, lat = f['geometry']['coordinates']
        p = f['properties']
        rows.append(dict(id=next_id, lon=lon, lat=lat, base_year=t0,
                         label=int(p['label']), grass_type=int(p['grass_type'])))
        next_id += 1

points_df = pd.DataFrame(rows)
before = len(points_df)
points_df = points_df.drop_duplicates(subset=['lon', 'lat', 'base_year']).reset_index(drop=True)
print('duplicates removed:', before - len(points_df))
print(points_df.groupby(['base_year', 'grass_type', 'label']).size().unstack(fill_value=0))

base year 2019: 750 points
base year 2020: 750 points
duplicates removed: 0
label                   0    1
base_year grass_type          
2019      0           250    0
          1           250  250
2020      0           250    0
          1           250  250


If a stratum comes back well below `POINTS_PER_STRATUM_PER_YEAR` (natural-grassland conversions are likely rare in this AOI), that is a finding in itself: report it rather than padding with other years or areas.

In [6]:
# ---- Feature extraction helpers ------------------------------------------------
def chunks(df, n=400):
    for i in range(0, len(df), n):
        yield df.iloc[i:i + n]

def to_fc(df):
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([r.lon, r.lat]), {'id': int(r.id)}) for r in df.itertuples()
    ])

def reduce_at_points(image, df, scale, retries=3):
    """Return {id: {band: value}} for the points in df, sampled from `image` (Reducer.first)."""
    out = {}
    for part in chunks(df):
        for attempt in range(retries):
            try:
                res = image.reduceRegions(to_fc(part), ee.Reducer.first(), scale=scale, tileScale=4).getInfo()
                break
            except Exception as e:
                print('retry', attempt + 1, type(e).__name__, str(e)[:120])
                time.sleep(5 * (attempt + 1))
        else:
            raise RuntimeError('reduceRegions failed repeatedly')
        for f in res['features']:
            out[f['properties']['id']] = f['properties']
    return out

AE_BANDS = [f'A{i:02d}' for i in range(64)]
ae_col = ee.ImageCollection(AE_ASSET)

def ae_image(year):
    return ae_col.filterDate(f'{year}-01-01', f'{year + 1}-01-01').mosaic().select(AE_BANDS)

srtm = ee.Image('USGS/SRTMGL1_003')
elevation = srtm.rename('elevation')
slope = ee.Terrain.slope(srtm).rename('slope')
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').select('precipitation')

def covariate_image(t0):
    y0 = t0 - WINDOW + 1
    precip = chirps.filterDate(f'{y0}-01-01', f'{t0 + 1}-01-01').sum().divide(WINDOW).rename('precip_mm_yr')
    crop_share = (isin(lulc(t0), CROP_CLASSES)
                  .focal_mean(radius=CROP_NEIGHBORHOOD_M, kernelType='circle', units='meters')
                  .rename('crop_share_2km'))
    return elevation.addBands(slope).addBands(precip).addBands(crop_share)

COV_NAMES = ['elevation', 'slope', 'precip_mm_yr', 'crop_share_2km']

In [7]:
X_seq = np.full((len(points_df), WINDOW, 64), np.nan, dtype=np.float32)
cov = np.full((len(points_df), len(COV_NAMES)), np.nan, dtype=np.float32)
id_to_row = {pid: i for i, pid in enumerate(points_df['id'])}

for t0 in BASE_YEARS:
    sub = points_df[points_df['base_year'] == t0]
    # static covariates (as of t0)
    res = reduce_at_points(covariate_image(t0), sub, scale=30)
    for pid, props in res.items():
        cov[id_to_row[pid]] = [props.get(n, np.nan) if props.get(n) is not None else np.nan for n in COV_NAMES]
    # AlphaEarth sequence t0-W+1 .. t0
    for j, year in enumerate(range(t0 - WINDOW + 1, t0 + 1)):
        res = reduce_at_points(ae_image(year), sub, scale=10)
        for pid, props in res.items():
            vals = [props.get(b) for b in AE_BANDS]
            if all(v is not None for v in vals):
                X_seq[id_to_row[pid], j] = vals
    print(f'base year {t0}: features extracted')

ok = ~np.isnan(X_seq).any(axis=(1, 2)) & ~np.isnan(cov).any(axis=1)
print(f'complete rows: {ok.sum()} / {len(ok)}')

base year 2019: features extracted
base year 2020: features extracted
complete rows: 1500 / 1500


In [8]:
valid_df = points_df[ok].reset_index(drop=True).copy()
X_seq, cov = X_seq[ok], cov[ok]

valid_df['block_id'] = (
    (valid_df['lon'] / GRID_DEG).apply(np.floor).astype(int).astype(str) + '_' +
    (valid_df['lat'] / GRID_DEG).apply(np.floor).astype(int).astype(str)
)

def assign_blocks_to_splits(df, seed=SEED, target=(0.70, 0.15, 0.15)):
    """Greedy block-level 70/15/15 split (same logic as 01_dados.ipynb): no block appears in two splits."""
    rng = np.random.default_rng(seed)
    block_ids = np.array(df['block_id'].unique())
    rng.shuffle(block_ids)
    names = ['train', 'val', 'test']
    counts = {n: 0 for n in names}
    sizes = df.groupby('block_id').size().to_dict()
    assignment = {}
    for bid in block_ids:
        total = sum(counts.values()) or 1
        deficits = {n: target[i] - counts[n] / total for i, n in enumerate(names)}
        chosen = max(deficits, key=deficits.get)
        assignment[bid] = chosen
        counts[chosen] += sizes[bid]
    return df['block_id'].map(assignment)

valid_df['split'] = assign_blocks_to_splits(valid_df)
assert (valid_df.groupby('block_id')['split'].nunique() == 1).all(), 'spatial leakage: block in several splits'

print(valid_df.groupby('split').size())
print(valid_df.groupby(['split', 'grass_type', 'label']).size().unstack(fill_value=0))

split
test      221
train    1038
val       241
dtype: int64
label               0    1
split grass_type          
test  0           119    0
      1            41   61
train 0           374    0
      1           359  305
val   0             7    0
      1           100  134


C:\Users\gabri\AppData\Local\Temp\ipykernel_8724\3428623401.py:13: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(block_ids)


In [9]:
import os
os.makedirs('../data/processed', exist_ok=True)

np.savez(
    '../data/processed/conversion_dataset.npz',
    X_seq=X_seq,                       # (N, WINDOW, 64) AlphaEarth embeddings, years t0-W+1..t0
    cov=cov, cov_names=np.array(COV_NAMES),
    y=valid_df['label'].to_numpy(),    # 1 = converted to cropland by t0+K, 0 = stayed grass
    grass_type=valid_df['grass_type'].to_numpy(),   # 0 = natural grassland (12), 1 = pasture (15)
    base_year=valid_df['base_year'].to_numpy(),
    split=valid_df['split'].to_numpy(),
    block_id=valid_df['block_id'].to_numpy(),
    lat=valid_df['lat'].to_numpy(), lon=valid_df['lon'].to_numpy(),
    point_id=valid_df['id'].to_numpy(),
    horizon=np.array(HORIZON), window=np.array(WINDOW),
)
valid_df.to_csv('../data/processed/conversion_points.csv', index=False)
print('saved. X_seq:', X_seq.shape, '| cov:', cov.shape)

saved. X_seq: (1500, 3, 64) | cov: (1500, 4)


## Sanity baseline

Majority class vs. logistic regression on mean embedding + covariates, then the same without `crop_share_2km`. If the neighborhood feature carries most of the signal, the model is mostly learning "next to existing cropland" (spatial diffusion) rather than land characteristics. Report both.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

d = np.load('../data/processed/conversion_dataset.npz', allow_pickle=True)
split, y, cov_names = d['split'], d['y'], list(d['cov_names'])
emb_mean = d['X_seq'].mean(axis=1)

def evaluate(name, feats, model):
    tr, te = split == 'train', split == 'test'
    model.fit(feats[tr], y[tr])
    pred = model.predict(feats[te])
    auc = roc_auc_score(y[te], model.predict_proba(feats[te])[:, 1]) if hasattr(model, 'predict_proba') and len(set(y[te])) > 1 else float('nan')
    print(f'{name:38s} acc={accuracy_score(y[te], pred):.3f}  macroF1={f1_score(y[te], pred, average="macro"):.3f}  AUC={auc:.3f}')

keep = [i for i, n in enumerate(cov_names) if n != 'crop_share_2km']
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))

evaluate('majority class', emb_mean, DummyClassifier(strategy='most_frequent'))
evaluate('LR: embedding only', emb_mean, lr())
evaluate('LR: embedding + terrain/rain', np.hstack([emb_mean, d['cov'][:, keep]]), lr())
evaluate('LR: + crop_share_2km', np.hstack([emb_mean, d['cov']]), lr())

ImportError: DLL load failed while importing _special_ufuncs: Uma política de Controle de Aplicativo bloqueou este arquivo.